# Quering the SOSA graph

In [25]:
# imports

import json
import time
import pandas as pd

from rdflib import Graph
import uuid
from rdflib import Namespace
from rdflib import FOAF, DCTERMS, SKOS, VANN, TIME, RDF, RDFS, XSD, OWL, Literal

In [26]:
g = Graph()
g.parse('/home/brieuc/Documents/dev/abromics/abromics-kg/dev/analysis/sosa-graph.ttl', format='ttl')

<Graph identifier=N70ea8fba66c84bf18e31873e8266f847 (<class 'rdflib.graph.Graph'>)>

## Quering the sosa graph

Check the journal to see all the queries to test on the SOSA graph

- Get all the resistance genes for a sample
- Rank the resistance genes found for a sample by the coverage percentage
- Get all the antibiotic, a specific strain is resistance to
- Get all the samples where a specific strain is present (filtering by mlst and by specie name)
- Get uniprot id of an antitbiotic and as well as it's molecular formula and a link to it's 3d structure
- Get the most spread resistance genes present in a certain geospacial area

In [27]:
## Get all the resistance gen for a sample

query = """
    PREFIX sosa: <http://www.w3.org/ns/sosa/>
    PREFIX schema: <https://schema.org/>
    PREFIX abromics: <https://abromics.fr/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    PREFIX obo: <http://purl.obolibrary.org/obo/>
    PREFIX aro: <http://purl.obolibrary.org/obo/aro.owl#>
    PREFIX go: <http://purl.org/obo/owl/GO#>
    PREFIX uniprot: <http://purl.uniprot.org/uniprot/>
    PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>

    SELECT DISTINCT ?resGeneNames
    WHERE {
        ?sample a sosa:FeatureOfInterest .
        ?sample schema:identifier "0001" .
        ?resGenesProperty a sosa:ObservableProperty .
        ?resGenesProperty rdfs:label "Antibiotic resistant gene names"^^xsd:string .
        ?observations sosa:hasObservableProperty ?resGenesProperty .
        ?observations sosa:hasFeatureOfInterest ?sample .
        ?observations sosa:hasResult ?results .
        ?results sosa:hasValue ?resGeneNames .
    }
"""

res = g.query(query)

for row in res:
    print(f"{row.resGeneNames}")

blaTEM-1A
tet(A)
sul1
qacE
dfrA1
aadA1


In [28]:
## Get all the resistance genes for a sample (with id = 0001) and rank them with their according to their coverage

query = """
    PREFIX sosa: <http://www.w3.org/ns/sosa/>
    PREFIX schema: <https://schema.org/>
    PREFIX abromics: <https://abromics.fr/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    PREFIX obo: <http://purl.obolibrary.org/obo/>
    PREFIX aro: <http://purl.obolibrary.org/obo/aro.owl#>
    PREFIX go: <http://purl.org/obo/owl/GO#>
    PREFIX uniprot: <http://purl.uniprot.org/uniprot/>
    PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>
    
    SELECT ?genes ?geneCoverage 
    WHERE {
        ?sample a sosa:FeatureOfInterest .
        ?sample schema:identifier "0001" .
        ?resGenesProperty a sosa:ObservableProperty .
        ?resGenesProperty rdfs:label "Antibiotic resistant gene coverage"^^xsd:string .
        ?observations sosa:hasObservableProperty ?resGenesProperty .
        ?observations sosa:hasFeatureOfInterest ?sample .
        ?genes a sosa:FeatureOfInterest .
        ?genes a go:Gene .
        ?observations sosa:hasFeatureOfInterest ?genes .
        ?observations sosa:hasResult ?results .
        ?results sosa:hasValue ?geneCoverage .
    }
    ORDER BY DESC(?geneCoverage)
"""

res = g.query(query)

for row in res:
    print(f"{row.genes} - {row.geneCoverage}")

file:///home/brieuc/Documents/dev/abromics/abromics-kg/dev/analysis/gene_1 - 100
file:///home/brieuc/Documents/dev/abromics/abromics-kg/dev/analysis/gene_5 - 100
file:///home/brieuc/Documents/dev/abromics/abromics-kg/dev/analysis/gene_3 - 97.95
file:///home/brieuc/Documents/dev/abromics/abromics-kg/dev/analysis/gene_2 - 97.8
file:///home/brieuc/Documents/dev/abromics/abromics-kg/dev/analysis/gene_4 - 84.68
file:///home/brieuc/Documents/dev/abromics/abromics-kg/dev/analysis/gene_6 - 76.17


In [29]:
## Get all the antibiotics a specific specie (defined by its ncbi taxonomic id) is resistant to

query = """
    PREFIX sosa: <http://www.w3.org/ns/sosa/>
    PREFIX schema: <https://schema.org/>
    PREFIX abromics: <https://abromics.fr/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    PREFIX obo: <http://purl.obolibrary.org/obo/>
    PREFIX aro: <http://purl.obolibrary.org/obo/aro.owl#>
    PREFIX go: <http://purl.org/obo/owl/GO#>
    PREFIX uniprot: <http://purl.uniprot.org/uniprot/>
    PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>

    SELECT DISTINCT ?antibiotics 
    WHERE {
        ?strain a aro:Strain ;
            a sosa:FeatureOfInterest .
        ?strain obo:RO_0002162 <http://purl.uniprot.org/taxonomy/562> . # Get the E.Coli taxon from ncbi
        ?strain aro:confers_resistance_to_antibiotic ?antibiotics
    }
"""

res = g.query(query)

for row in res:
    print(f"{row.antibiotics}")

http://purl.obolibrary.org/obo/aro.owl#ampicillin
http://purl.obolibrary.org/obo/aro.owl#tetracycline
http://purl.obolibrary.org/obo/aro.owl#sulfisoxazole
http://purl.obolibrary.org/obo/aro.owl#trimethoprim
http://purl.obolibrary.org/obo/aro.owl#streptomycin
http://purl.obolibrary.org/obo/aro.owl#chloramphenicol
http://purl.obolibrary.org/obo/aro.owl#tcolistin
http://purl.obolibrary.org/obo/aro.owl#kanamycin


In [43]:
## Find all the strain with the same antibiotic resistance gene found

## change "sul1" to "sul2" to showcase that the filtering process is working as the strain 1
## contains "sul1" but not "sul2" 

query = """
    PREFIX sosa: <http://www.w3.org/ns/sosa/>
    PREFIX schema: <https://schema.org/>
    PREFIX abromics: <https://abromics.fr/>
    PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
    PREFIX obo: <http://purl.obolibrary.org/obo/>
    PREFIX aro: <http://purl.obolibrary.org/obo/aro.owl#>
    PREFIX go: <http://purl.org/obo/owl/GO#>
    PREFIX uniprot: <http://purl.uniprot.org/uniprot/>
    PREFIX geo: <http://www.w3.org/2003/01/geo/wgs84_pos#>
    
    SELECT ?strains ?latitude ?longitude ?altitude
    WHERE {
        ?strains a aro:Strain .
            
        ?gene rdfs:label "sul1" .
        ?geneObservableProperty a sosa:ObservableProperty ;
            rdfs:label "Antibiotic resistant gene names"^^xsd:string .
        ?observations sosa:hasFeatureOfInterest ?gene ;
            sosa:hasObservableProperty ?geneObservableProperty ;
            sosa:hasFeatureOfInterest ?strains ;
            sosa:hasFeatureOfInterest ?location .
        ?location geo:lat ?latitude ;
            geo:long ?longitude ;
            geo:alt ?altitude .
    }
"""

res = g.query(query)

for row in res:
    print(f"{row.strains} - {row.latitude} - {row.longitude} - {row.altitude}")

file:///home/brieuc/Documents/dev/abromics/abromics-kg/dev/analysis/strain_1 - 35.8648067 - -120.6195831 - 12.75
file:///home/brieuc/Documents/dev/abromics/abromics-kg/dev/analysis/strain_2 - 47.209110 - -1.553479 - 10.00
